In [3]:
import numpy as np

# 假设 bs 和 B 的值
bs = 1
B =1
size = 8
bit_size = 3

vector1 = np.random.rand(bs, 2, B, size, size, 5)
vector2 = np.random.randint(-2**(bit_size-1)+1, 2**(bit_size-1), size=(bs, 2, B, size, size, 5))
vector3 = np.zeros((bs, size, size, 2**bit_size))

# 广播和高级索引
vector1_transposed = np.transpose(vector1, (0, 3, 4, 1, 2, 5))
vector2_transposed = np.transpose(vector2, (0, 3, 4, 1, 2, 5))
print(vector1.shape, " to ", vector1_transposed.shape)
print(vector2.shape, " to ", vector2_transposed.shape)

vector1_reshaped = vector1_transposed.reshape(bs, size, size, -1)
vector2_reshaped = vector2_transposed.reshape(bs, size, size, -1)
print(vector1_transposed.shape, " to ", vector1_reshaped.shape)
print(vector2_transposed.shape, " to ", vector2_reshaped.shape)

# 将 vector2 的索引值调整为非负
vector2_reshaped = vector2_reshaped + 2**(bit_size-1)-1

np.add.at(vector3, (np.arange(bs)[:, None, None, None], np.arange(size)[None, :, None, None], np.arange(size)[None, None, :, None], vector2_reshaped), vector1_reshaped)
print(vector1_reshaped[0,0,1,:])
print(vector2_reshaped[0,0,1,:])
print(vector3[0,0,1,:])

(1, 2, 1, 8, 8, 5)  to  (1, 8, 8, 2, 1, 5)
(1, 2, 1, 8, 8, 5)  to  (1, 8, 8, 2, 1, 5)
(1, 8, 8, 2, 1, 5)  to  (1, 8, 8, 10)
(1, 8, 8, 2, 1, 5)  to  (1, 8, 8, 10)
[0.54968092 0.85263745 0.37725468 0.13756578 0.08002029 0.49297515
 0.39173948 0.74041119 0.10728626 0.07341904]
[3 1 0 5 4 0 2 4 5 1]
[0.87022983 0.92605649 0.39173948 0.54968092 0.82043148 0.24485204
 0.         0.        ]


In [48]:
import numpy as np

def diagonal_sum(a, b):
    a = np.array(a)
    b = np.array(b)
    
    n = len(a)
    
    P = np.outer(a, b)
    print(P)
    
    diagonals_sum = [np.sum(np.diag(P, k)) for k in range(-n+1, n)]
    
    return diagonals_sum

a = [0.1, 0.2, 0.3, 0.4, 0.5]
b = [0.5, 0.4, 0.3, 0.2, 0.1]

result = diagonal_sum(a, b)
print("Total diagonal sum:", result)
print("numpy.convolve:",np.convolve(a[::-1], b, mode='full'))

[[0.05 0.04 0.03 0.02 0.01]
 [0.1  0.08 0.06 0.04 0.02]
 [0.15 0.12 0.09 0.06 0.03]
 [0.2  0.16 0.12 0.08 0.04]
 [0.25 0.2  0.15 0.1  0.05]]
Total diagonal sum: [0.25, 0.4, 0.4600000000000001, 0.43999999999999995, 0.35000000000000003, 0.2, 0.1, 0.04000000000000001, 0.010000000000000002]
numpy.convolve: [0.25 0.4  0.46 0.44 0.35 0.2  0.1  0.04 0.01]


In [38]:
import numpy as np

def diagonal_sum_1(a, b):
    a = np.array(a)
    b = np.array(b)
    
    n = len(a)
    
    diagonals_sum = []
    for k in range(-n + 1, n):
        diagonals_sum.append(np.sum(a[max(0, -k):n - max(0, k)] * b[max(0, k):n - max(0, -k)]))
    
    return diagonals_sum

a = [0.1, 0.2, 0.3, 0.4, 0.5]
b = [0.5, 0.4, 0.3, 0.2, 0.1]

result = diagonal_sum_1(a, b)
print("Total diagonal sum:", result)
print("numpy.convolve:",np.convolve(a, b))

Total diagonal sum: [0.25, 0.4, 0.4600000000000001, 0.43999999999999995, 0.35000000000000003, 0.2, 0.1, 0.04000000000000001, 0.010000000000000002]
numpy.convolve: [0.05 0.14 0.26 0.4  0.55 0.4  0.26 0.14 0.05]


In [74]:
import numpy as np
import torch
import torchaudio
import torchaudio.functional as F

def diagonal_sum_2(a, b):
    a = np.array(a)
    b = np.array(b)
    
    bs, ch, B, n = a.shape
    result_shape = (bs, ch, B, 2 * n - 1)
    diagonals_sum = np.zeros(result_shape)
    
    for b_idx in range(B):
        for k in range(-n + 1, n):
            diag_sum = np.sum(
                a[:, :, b_idx, max(0, -k):n - max(0, k)] *
                b[:, :, b_idx, max(0, k):n - max(0, -k)],
                axis=-1
            )
            diagonals_sum[:, :, b_idx, k + n - 1] = diag_sum
    
    return diagonals_sum

# Example usage
a = np.random.rand(2, 3, 4, 5)
b = np.random.rand(2, 3, 4, 5)
result = diagonal_sum_2(a, b)

print("ab:", diagonal_sum_1(a[0,0,0,:], b[0,0,0,:]))
print("results[0,0,0,:]:", result[0,0,0,:])
print("results shape:", result.shape)  # Should print (2, 3, 4, 9)
# print("results",result)

a_tensor = torch.from_numpy(a[:,:,:,::-1].copy()).double()
b_tensor = torch.from_numpy(b).double()

convolved_tensor = F.convolve(a_tensor, b_tensor)
print("results shape:", convolved_tensor.shape)
print("convolved_tensor:", convolved_tensor[0,0,0,:])

ab: [0.20263990254985909, 0.7086499033244426, 1.0685464233337578, 1.4127325309523253, 1.8874676764828258, 1.5536695505499587, 1.0334567577384919, 0.8984417314934039, 0.3193103611626825]
results[0,0,0,:]: [0.2026399  0.7086499  1.06854642 1.41273253 1.88746768 1.55366955
 1.03345676 0.89844173 0.31931036]
results shape: (2, 3, 4, 9)
results shape: torch.Size([2, 3, 4, 9])
convolved_tensor: tensor([0.20263990, 0.70864990, 1.06854642, 1.41273253, 1.88746768, 1.55366955,
        1.03345676, 0.89844173, 0.31931036], dtype=torch.float64)


In [35]:
import numpy as np

def diagonal_sum(a, b):
    a = np.array(a)
    b = np.array(b)
    
    bs, ch, B, n = a.shape
    result_shape = (bs, ch, B, 2 * n - 1)
    
    # Apply convolution along the last axis using broadcasting
    diagonals_sum = np.apply_along_axis(
        lambda x: np.convolve(x[0], x[1], mode='full'), 
        -1, 
        np.stack((a, b), axis=-1)
    )
    
    return diagonals_sum

# Example usage
a = np.random.rand(2, 3, 4, 5)
b = np.random.rand(2, 3, 4, 5)
result = diagonal_sum(a, b)
print(result.shape)  # Should print (2, 3, 4, 9)


(2, 3, 4, 5, 1)


In [73]:
a = np.random.rand(5)
b = torch.from_numpy(a)
print("a",a)
torch.set_printoptions(profile="full")
torch.set_printoptions(threshold=float('inf'), precision=8)
print("b",b)

a [0.64867891 0.47492326 0.26037906 0.39767643 0.56810437]
b tensor([0.64867891, 0.47492326, 0.26037906, 0.39767643, 0.56810437],
       dtype=torch.float64)


In [37]:
# 假设你的张量是一个 NumPy 数组
tensor = np.random.randint(0, 10,(1, 2, 3, 5))
# 将B轴右移一位
shifted_tensor = np.roll(tensor, shift=1, axis=2)
shifted_tensor[:,:,0,:] = [1,0,0,0,0]

print(tensor)
print(shifted_tensor)

[[[[7 6 6 6 1]
   [0 5 5 2 1]
   [9 1 1 3 6]]

  [[3 4 2 0 5]
   [4 3 3 1 4]
   [5 1 8 8 0]]]]
[[[[1 0 0 0 0]
   [7 6 6 6 1]
   [0 5 5 2 1]]

  [[1 0 0 0 0]
   [3 4 2 0 5]
   [4 3 3 1 4]]]]


In [13]:
import numpy as np
# 创建两个形状为 (2, 3, 4, 5) 的示例向量
vector1 = torch.randint(0, 10, (2,3,4,5))
vector2 = torch.randint(0, 10, (2,3,4,5))

def get_diff_indexes(indexes, indexes_shifted):
    middle = indexes[:,:,:,0] - indexes_shifted[:,:,:,0] # shape: [bs,ch,B]
    diff_indexes = middle.unsqueeze(-1) # shape: [bs,ch,B,1]
    diff_indexes = diff_indexes.expand(-1, -1, -1, 2*5-1) + np.arange(-5 + 1,5) # shape: [bs,ch,B,9]
    return diff_indexes

print(get_diff_indexes(vector1,vector2).shape)

torch.Size([2, 3, 4, 9])


In [14]:
import torch
import torchaudio
import torchaudio.functional as F

# 定义两个vector A和B
a = torch.randint(0, 10, (5,))
b = torch.randint(0, 10, (5,))

c = F.convolve(a,b)
print(a)
print(b)
print(c)

tensor([7, 6, 9, 9, 7])
tensor([8, 3, 7, 7, 7])
tensor([ 56,  69, 139, 190, 237, 189, 175, 112,  49])


In [24]:
import torch

# Example tensors
p_lum_dc = torch.rand(5, 5)  # Example tensor
num_blocks = 2  # Scalar tensor, division will be broadcasted

# Correct division
p_lum_dc /= num_blocks

print(p_lum_dc)


tensor([[0.2694, 0.2742, 0.1282, 0.0921, 0.1945],
        [0.3864, 0.0458, 0.2120, 0.2517, 0.1402],
        [0.3600, 0.4894, 0.3283, 0.2249, 0.4895],
        [0.3257, 0.3505, 0.2917, 0.4351, 0.2629],
        [0.1643, 0.3785, 0.0573, 0.1682, 0.0082]])
